# Top 10 Projections by Position (Linear Regression)

This notebook uses the position-specific linear regression models from `linear_regression.ipynb` to project the top 10 QBs, RBs, WRs and TEs, then checks how accurate those projections were against what actually happened.

**How the test works (a true backtest):**
1. Fit each position's model on **2024** data only (same features as `linear_regression.ipynb`).
2. Feed each player's **2024 per-game usage** into that model to project their **2025** PPR points per game.
3. Compare the projected top 10 against the **actual 2025** top 10.

The model never sees 2025 data, so this measures real predictive accuracy rather than in-sample fit. As a sanity check, every result is compared with a naive baseline: *"a player will score the same PPR/game next year as he did this year."*

The final section refits the models on 2025 data and projects the top 10s for 2026.

Run from the repository root, like the other notebooks.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from scipy.stats import spearmanr
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

df = pd.read_csv("DataSets/FantasyData.csv")

count_cols = ['G', 'GS', 'RushAtt', 'RushYds', 'RushTD', 'Rush1D',
              'Cmp', 'PassAtt', 'PassYds', 'PassTD', 'Int', 'Pass1D',
              'Tgt', 'Rec', 'ReceiveYds', 'ReceiveTD', 'Receive1D']
df[count_cols] = df[count_cols].fillna(0)
df = df[df['Pos'].isin(['QB', 'RB', 'WR', 'TE'])].copy()
df = df.sort_values('G', ascending=False).drop_duplicates(['Player', 'year']).sort_index()

g0 = df['G'].replace(0, np.nan)
df['Plays'] = df['RushAtt'] + df['Rec'] + df['PassAtt']
df['TgtPG'] = df['Tgt'] / g0
df['RushAttPG'] = df['RushAtt'] / g0
df['PassAttPG'] = df['PassAtt'] / g0
df['TDPG'] = (df['RushTD'] + df['ReceiveTD']) / g0

MIN_PLAYS = {'QB': 200, 'RB': 100, 'WR': 40, 'TE': 30}
qual = df[df['Plays'] >= df['Pos'].map(MIN_PLAYS)].copy()

# Same position-specific features as linear_regression.ipynb
FEATURES = {
    'QB': ['PassAttPG', 'RushAttPG'],
    'RB': ['RushAttPG', 'TgtPG', 'TDPG'],
    'WR': ['TgtPG'],
    'TE': ['TgtPG'],
}
POSITIONS = ['QB', 'RB', 'WR', 'TE']

# "Actual" top 10 = best PPR points per game, min 8 games (filters out 1-2 game flukes)
MIN_GAMES_ACTUAL = 8
print(qual.groupby(['year', 'Pos']).size().unstack())

Pos   QB  RB  TE  WR
year                
2024  40  51  37  71
2025  37  54  40  60


**What this shows:** the number of qualified player-seasons per position and year after applying the minimum-plays filter. Each season has roughly 40 QBs, 50 RBs, 40 TEs and 60–70 WRs, which is enough to fit a simple regression per position but small enough that a few unusual seasons can move the coefficients.

In [3]:
def fit_and_project(pos, train_year):
    """Fit the position model on one season and project PPR/game from that season's usage."""
    train = qual[(qual['Pos'] == pos) & (qual['year'] == train_year)].copy()
    model = LinearRegression().fit(train[FEATURES[pos]], train['FantasyPPRPerGame'])
    train['ProjPPG'] = model.predict(train[FEATURES[pos]])
    return model, train.set_index('Player')

actual_2025 = df[df['year'] == 2025].set_index('Player')
results, tables = {}, {}

for pos in POSITIONS:
    model, proj = fit_and_project(pos, 2024)

    actual = (df[(df['year'] == 2025) & (df['Pos'] == pos) & (df['G'] >= MIN_GAMES_ACTUAL)]
              .sort_values('FantasyPPRPerGame', ascending=False))
    top_actual = list(actual['Player'].head(10))
    top_model = list(proj.sort_values('ProjPPG', ascending=False).index[:10])
    top_naive = list(proj.sort_values('FantasyPPRPerGame', ascending=False).index[:10])

    # Players with a 2024 projection who also played 8+ games in 2025
    both = proj.join(actual_2025[['FantasyPPRPerGame', 'G']], rsuffix='_2025', how='inner')
    both = both[both['G_2025'] >= MIN_GAMES_ACTUAL]

    results[pos] = {
        'Model hits (of 10)': len(set(top_model) & set(top_actual)),
        'Naive hits (of 10)': len(set(top_naive) & set(top_actual)),
        'Model rank corr': spearmanr(both['ProjPPG'], both['FantasyPPRPerGame_2025'])[0],
        'Naive rank corr': spearmanr(both['FantasyPPRPerGame'], both['FantasyPPRPerGame_2025'])[0],
        'Model MAE (PPG)': (both['ProjPPG'] - both['FantasyPPRPerGame_2025']).abs().mean(),
        'Naive MAE (PPG)': (both['FantasyPPRPerGame'] - both['FantasyPPRPerGame_2025']).abs().mean(),
        'Players compared': len(both),
    }

    rows = []
    for i, p in enumerate(top_model, 1):
        a = actual_2025.loc[p] if p in actual_2025.index else None
        played = a is not None and a['G'] >= MIN_GAMES_ACTUAL
        actual_rank = top_actual.index(p) + 1 if p in top_actual else None
        if not played:
            act_rank_str = "<8 games" if a is not None else "didn't play"
        else:
            act_rank_str = str(actual_rank) if actual_rank else "outside top 10"
        rows.append({
            'Proj rank': i, 'Player': p,
            'Proj PPG': round(proj.loc[p, 'ProjPPG'], 1),
            'Actual 2025 PPG': round(a['FantasyPPRPerGame'], 1) if a is not None else None,
            'Actual rank': act_rank_str,
            'Hit': '✓' if p in top_actual else '',
        })
    tables[pos] = (pd.DataFrame(rows), top_actual)

summary = pd.DataFrame(results).T.round(2)
summary

,Model hits (of 10),Naive hits (of 10),Model rank corr,Naive rank corr,Model MAE (PPG),Naive MAE (PPG),Players compared
QB,4.0,4.0,0.32,0.39,2.85,3.17,28.0
RB,5.0,6.0,0.72,0.74,3.24,3.16,42.0
WR,4.0,4.0,0.66,0.64,3.09,3.34,61.0
TE,5.0,6.0,0.48,0.41,3.06,3.34,36.0


**What this means:**

- **Top-10 hits:** the regression got **18 of 40** top-10 players right across the four positions. The naive "same PPR/game as last year" guess got **20 of 40**, so at picking the top 10 the model is about as good as simply using last year's results, not better.
- **Rank correlation** (how well the full ordering of players matched, 1.0 = perfect): RB (0.72) and WR (0.66) are fairly predictable year to year; TE (0.48) is moderate; QB (0.32) is weak.
- **MAE** (average miss in PPR points per game): the model misses by about 3 points per game. It beats the naive guess at QB, WR and TE because it pulls fluky high or low seasons back toward what the player's volume supports.
- Takeaway: usage explains a season very well (the R² of 0.8–0.9 in `linear_regression.ipynb`), but last season's usage is only a moderate predictor of next season's points.

In [5]:
t, top_actual = tables['QB']
print("=== QB: projected 2025 top 10 (from 2024 usage) ===")
print(t.to_string(index=False))
print()
print("Actual 2025 top 10:", ", ".join(f"{i}. {p}" for i, p in enumerate(top_actual, 1)))
missed = [p for p in top_actual if p not in set(t['Player'])]
print("Missed by the model:", ", ".join(missed))

=== QB: projected 2025 top 10 (from 2024 usage) ===
 Proj rank             Player  Proj PPG  Actual 2025 PPG    Actual rank Hit
         1       Daniel Jones      21.2             17.7 outside top 10    
         2     Jayden Daniels      20.8             16.6       <8 games    
         3        Jalen Hurts      20.4             19.2              7   ✓
         4      Lamar Jackson      20.0             16.8 outside top 10    
         5             Bo Nix      19.2             17.7 outside top 10    
         6    Patrick Mahomes      18.4             20.1              5   ✓
         7     Caleb Williams      18.2             18.8              8   ✓
         8         Joe Burrow      17.9             16.8 outside top 10    
         9 Anthony Richardson      17.6              1.1       <8 games    
        10         Josh Allen      17.5             21.7              1   ✓

Actual 2025 top 10: 1. Josh Allen, 2. Matthew Stafford, 3. Drake Maye, 4. Trevor Lawrence, 5. Patrick Mahomes, 

**What this means (QB, 4/10):** the model's QB ranking is driven by pass attempts and rush attempts per game, so it rewards volume and running. That overrated high-volume or rushing QBs like Daniel Jones, Lamar Jackson and Bo Nix, and completely missed efficient pocket passers like Matthew Stafford, Jared Goff and Dak Prescott, who don't throw more often but get more out of each attempt. Two of its picks (Jayden Daniels, Anthony Richardson) missed most of 2025 to injury, which no usage-based model can see coming. QB is the weakest position for this model; adding efficiency stats (yards per attempt, TD rate) would likely help most here.

In [7]:
t, top_actual = tables['RB']
print("=== RB: projected 2025 top 10 (from 2024 usage) ===")
print(t.to_string(index=False))
print()
print("Actual 2025 top 10:", ", ".join(f"{i}. {p}" for i, p in enumerate(top_actual, 1)))
missed = [p for p in top_actual if p not in set(t['Player'])]
print("Missed by the model:", ", ".join(missed))

=== RB: projected 2025 top 10 (from 2024 usage) ===
 Proj rank             Player  Proj PPG  Actual 2025 PPG    Actual rank Hit
         1       Jahmyr Gibbs      20.2             21.7              3   ✓
         2     Bijan Robinson      19.6             22.0              2   ✓
         3     Saquon Barkley      19.4             14.5 outside top 10    
         4       Alvin Kamara      19.2              9.5 outside top 10    
         5     Kyren Williams      19.0             15.7 outside top 10    
         6          Joe Mixon      18.5              NaN    didn't play    
         7    Jonathan Taylor      18.1             21.3              4   ✓
         8        Josh Jacobs      17.8             16.1              9   ✓
         9      Derrick Henry      17.7             16.8              7   ✓
        10 Kenneth Walker III      17.5             11.3 outside top 10    

Actual 2025 top 10: 1. Christian McCaffrey, 2. Bijan Robinson, 3. Jahmyr Gibbs, 4. Jonathan Taylor, 5. De'Von A

**What this means (RB, 5/10):** the RB model did best at the top, nailing Bijan Robinson, Jahmyr Gibbs and Jonathan Taylor. Its biggest miss, Christian McCaffrey (the actual #1), played only 4 games in 2024 and didn't meet the usage threshold, so the model had nothing to project him from. Joe Mixon didn't play in 2025, and Alvin Kamara and Kenneth Walker III fell off. Rookie Cam Skattebo had no 2024 season at all. Note that this model uses touchdowns per game as a feature, which carries forward touchdown luck from the previous season.

In [9]:
t, top_actual = tables['WR']
print("=== WR: projected 2025 top 10 (from 2024 usage) ===")
print(t.to_string(index=False))
print()
print("Actual 2025 top 10:", ", ".join(f"{i}. {p}" for i, p in enumerate(top_actual, 1)))
missed = [p for p in top_actual if p not in set(t['Player'])]
print("Missed by the model:", ", ".join(missed))

=== WR: projected 2025 top 10 (from 2024 usage) ===
 Proj rank           Player  Proj PPG  Actual 2025 PPG    Actual rank Hit
         1     Malik Nabers      21.1             14.3       <8 games    
         2    Ja'Marr Chase      19.1             19.7              3   ✓
         3      CeeDee Lamb      18.8             14.4 outside top 10    
         4    Davante Adams      18.7             15.9              9   ✓
         5       Puka Nacua      17.9             23.6              1   ✓
         6     Drake London      17.3             16.8              7   ✓
         7      Tee Higgins      16.9             14.0 outside top 10    
         8 Justin Jefferson      16.8             11.8 outside top 10    
         9   Garrett Wilson      16.8             14.2       <8 games    
        10     Chris Godwin      16.4              9.0 outside top 10    

Actual 2025 top 10: 1. Puka Nacua, 2. Jaxon Smith-Njigba, 3. Ja'Marr Chase, 4. Amon-Ra St. Brown, 5. Rashee Rice, 6. George Pickens, 

**What this means (WR, 4/10):** for receivers the model only uses targets per game, so it's essentially "last year's target share = next year's points." It correctly had Puka Nacua, Ja'Marr Chase, Drake London and Davante Adams. Its #1 pick, Malik Nabers, missed most of 2025 to injury. The misses were mostly players whose role grew: Jaxon Smith-Njigba and George Pickens (who moved from PIT to DAL) both saw big jumps in targets that 2024 usage couldn't predict.

In [11]:
t, top_actual = tables['TE']
print("=== TE: projected 2025 top 10 (from 2024 usage) ===")
print(t.to_string(index=False))
print()
print("Actual 2025 top 10:", ", ".join(f"{i}. {p}" for i, p in enumerate(top_actual, 1)))
missed = [p for p in top_actual if p not in set(t['Player'])]
print("Missed by the model:", ", ".join(missed))

=== TE: projected 2025 top 10 (from 2024 usage) ===
 Proj rank         Player  Proj PPG  Actual 2025 PPG    Actual rank Hit
         1   Trey McBride      14.6             18.6              1   ✓
         2   Brock Bowers      14.3             14.5              4   ✓
         3    David Njoku      14.1              7.2 outside top 10    
         4   Travis Kelce      13.4             11.2              9   ✓
         5    Evan Engram      11.8              6.4 outside top 10    
         6    Jonnu Smith      11.0              5.0 outside top 10    
         7  George Kittle      10.7             14.7              2   ✓
         8     Cade Otton      10.6              7.6 outside top 10    
         9 T.J. Hockenson      10.6              7.5 outside top 10    
        10  Jake Ferguson      10.5             11.2             10   ✓

Actual 2025 top 10: 1. Trey McBride, 2. George Kittle, 3. Tucker Kraft, 4. Brock Bowers, 5. Kyle Pitts, 6. Dallas Goedert, 7. Sam LaPorta, 8. Harold Fannin

**What this means (TE, 5/10):** the model correctly picked the top TE (Trey McBride) and had Brock Bowers, George Kittle, Travis Kelce and Jake Ferguson in the top 10. Its misses (David Njoku, Evan Engram, Jonnu Smith) all saw their production fall sharply in 2025. It couldn't foresee breakouts like Tucker Kraft and Kyle Pitts, or rookie Harold Fannin, who had no 2024 season.

## 2026 projections (using 2024 + 2025)

To project 2026, the models use **both** seasons:

1. **Fit:** each position's regression is trained on all qualified 2024 and 2025 player-seasons together (about twice the data of a single season).
2. **Project:** each player's per-game usage is combined across 2024 and 2025 (total attempts/targets/TDs divided by total games), so a player's two-year workload drives the projection rather than one season.

Only players who played in 2025 are included (retired or out-of-league players are dropped), and a player's 2025 position and team are used. The combined two-season plays must still meet the position's minimum plays. Rookies and offseason role changes aren't captured, and these projections can't be scored until 2026 data is added.

In [14]:
# 1. Fit each position model on 2024 + 2025 qualified seasons together
models_2026 = {}
for pos in POSITIONS:
    train = qual[(qual['Pos'] == pos) & (qual['year'].isin([2024, 2025]))]
    models_2026[pos] = LinearRegression().fit(train[FEATURES[pos]], train['FantasyPPRPerGame'])
    print(f"{pos}: trained on {len(train)} player-seasons")

# 2. Combine each player's 2024 + 2025 usage into two-year per-game rates
totals = ['G', 'Tgt', 'RushAtt', 'PassAtt', 'RushTD', 'ReceiveTD', 'Plays']
two_yr = df[df['year'].isin([2024, 2025])].groupby('Player')[totals].sum()
latest = df[(df['year'] == 2025) & (df['G'] > 0)].set_index('Player')[['Pos', 'Team']]
two_yr = two_yr.join(latest, how='inner')          # must have played in 2025
two_yr['TgtPG'] = two_yr['Tgt'] / two_yr['G']
two_yr['RushAttPG'] = two_yr['RushAtt'] / two_yr['G']
two_yr['PassAttPG'] = two_yr['PassAtt'] / two_yr['G']
two_yr['TDPG'] = (two_yr['RushTD'] + two_yr['ReceiveTD']) / two_yr['G']
two_yr = two_yr[two_yr['Plays'] >= two_yr['Pos'].map(MIN_PLAYS)]

QB: trained on 77 player-seasons
RB: trained on 105 player-seasons
WR: trained on 131 player-seasons
TE: trained on 77 player-seasons


**What this shows:** training on both seasons roughly doubles the data each model learns from (77–131 player-seasons per position instead of 37–71). Combining each player's 2024 and 2025 usage means a two-year workload drives his projection, which smooths out a single unusually good or bad year.

In [16]:
# 3. Project 2026 PPR/game and list the top 10 at each position
proj_2026 = {}
for pos in POSITIONS:
    pool = two_yr[two_yr['Pos'] == pos].copy()
    pool['Proj 2026 PPG'] = models_2026[pos].predict(pool[FEATURES[pos]]).round(1)
    top = (pool.sort_values('Proj 2026 PPG', ascending=False).head(10)
           .reset_index()[['Player', 'Team', 'G', 'Proj 2026 PPG']]
           .rename(columns={'G': 'Games 24-25'}))
    top['Games 24-25'] = top['Games 24-25'].astype(int)
    top.index = range(1, 11)
    proj_2026[pos] = top
    print(f"=== {pos}: projected 2026 top 10 (2024 + 2025 usage) ===")
    print(top.to_string())
    print()

=== QB: projected 2026 top 10 (2024 + 2025 usage) ===
             Player Team  Games 24-25  Proj 2026 PPG
1    Jayden Daniels  WAS           24           20.6
2            Bo Nix  DEN           34           19.9
3       Jalen Hurts  PHI           31           19.3
4   Patrick Mahomes  KAN           30           19.3
5    Caleb Williams  CHI           34           18.5
6      Kyler Murray  ARI           22           18.1
7      Daniel Jones  IND           23           17.9
8        Josh Allen  BUF           34           17.7
9     Lamar Jackson  BAL           30           17.4
10       Joe Burrow  CIN           25           17.2

=== RB: projected 2026 top 10 (2024 + 2025 usage) ===
                 Player Team  Games 24-25  Proj 2026 PPG
1   Christian McCaffrey  SFO           21           22.5
2          Jahmyr Gibbs  DET           34           20.6
3       Jonathan Taylor  IND           31           19.7
4        Bijan Robinson  ATL           34           19.6
5         De'Von Achane

**What this means:** these are the model's 2026 top 10s at each position. A few things to keep in mind:

- **Expect about 4–5 of each list to be right.** That's how the same method did when projecting 2025 from 2024.
- **QB carries the same bias** toward high-volume and rushing QBs (Jayden Daniels, Bo Nix, Kyler Murray, Daniel Jones near the top), and likely underrates efficient passers again.
- **One-season players are less reliable.** Rookies like Cam Skattebo and Harold Fannin, and players who missed lots of time like Rashee Rice (12 games over two years), are projected from one season or a small sample.
- **Offseason changes aren't captured.** Trades, free-agent moves, new coordinators and 2026 rookies are invisible to the model.
- Once 2026 data is added to `DataSets/FantasyData.csv`, these picks can be scored with the same backtest above.